# ArchDraft — Train Floor-Plan Models (Kaggle)

Two models for `src/05_parse_floorplan.py`:

1. **U-Net wall segmentation** — CVC FPP dataset (images + PNG masks)
2. **YOLOv8 door/symbol/text** — floor-plan detection dataset (optional)

## Setup

1. Kaggle → **GPU** on
2. **Add Input** → search **CVC FPP** or *Floor Plan* segmentation datasets
3. Run all cells
4. Download from **Output**:
   - `floorplan_unet.pt` → `models/floorplan_unet.pt`
   - `floorplan_yolo.pt` → `models/floorplan_yolo.pt` (optional)

## Validate locally

```powershell
python -m src.05_parse_floorplan data/raw_floorplans/your_plan.png --debug-mask data/wall_mask.png
```

In [ ]:
!pip install -q ultralytics segmentation-models-pytorch albumentations

In [ ]:
from pathlib import Path

INPUT = Path("/kaggle/input")
print("Input folders:")
for p in sorted(INPUT.iterdir()):
    print(" ", p.name)

# Set manually if auto-detect fails:
# CVC_ROOT = Path("/kaggle/input/cvc-fpp-or-similar")
CVC_ROOT = None
for folder in INPUT.iterdir():
    if not folder.is_dir():
        continue
    if list(folder.rglob("*mask*")) or list(folder.rglob("*.png")):
        CVC_ROOT = folder
        break

if CVC_ROOT is None:
    raise SystemExit("Add CVC FPP (or floor-plan mask dataset) as Kaggle Input")
print("CVC_ROOT =", CVC_ROOT)

In [ ]:
# --- Part 1: U-Net wall segmentation ---
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from pathlib import Path

class WallDataset(Dataset):
    def __init__(self, image_paths, mask_paths, size=256):
        self.images = image_paths
        self.masks = mask_paths
        self.size = size

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = cv2.imread(str(self.images[idx]))
        msk = cv2.imread(str(self.masks[idx]), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (self.size, self.size))
        msk = cv2.resize(msk, (self.size, self.size))
        img = torch.from_numpy(img.transpose(2, 0,  1)).float() / 255.0
        msk = torch.from_numpy((msk > 127).astype(np.float32)).unsqueeze(0)
        return img, msk

# TODO: pair image/mask paths from your CVC FPP folder layout
images = sorted(CVC_ROOT.rglob("*.jpg"))[:200]
masks = sorted(CVC_ROOT.rglob("*mask*.png"))[:200]
if len(images) < 10 or len(masks) < 10:
    print("Warning: few pairs found — adjust glob patterns for your dataset layout")
    pairs = min(len(images), len(masks), 50)
    images, masks = images[:pairs], masks[:pairs]

ds = WallDataset(images, masks)
dl = DataLoader(ds, batch_size=8, shuffle=True)

model = smp.Unet(encoder_name="resnet18", encoder_weights="imagenet", in_channels=3, classes=1)
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.BCEWithLogitsLoss()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

EPOCHS = 15
for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        opt.step()
        total += loss.item()
    print(f"epoch {epoch+1}/{EPOCHS} loss={total/len(dl):.4f}")

out_unet = Path("/kaggle/working/floorplan_unet.pt")
torch.save(model.state_dict(), out_unet)
print("Saved", out_unet)

In [ ]:
# --- Part 2: YOLO floor-plan symbols (doors, text regions) ---
# If you have a YOLO-format floor-plan dataset with data.yaml:
from ultralytics import YOLO
import yaml

yaml_path = None
for y in CVC_ROOT.rglob("data.yaml"):
    yaml_path = y
    break

if yaml_path:
    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)
    for key in ("train", "val"):
        if key in cfg and cfg[key] and not Path(cfg[key]).is_absolute():
            cfg[key] = str((yaml_path.parent / cfg[key]).resolve())
    fixed = Path("/kaggle/working/floorplan_data.yaml")
    with open(fixed, "w") as f:
        yaml.dump(cfg, f)

    yolo = YOLO("yolov8n.pt")
    yolo.train(data=str(fixed), epochs=30, imgsz=640, batch=16,
               project="/kaggle/working/runs/floorplan", name="train", exist_ok=True)
    best = "/kaggle/working/runs/floorplan/train/weights/best.pt"
    import shutil
    shutil.copy(best, "/kaggle/working/floorplan_yolo.pt")
    print("Saved floorplan_yolo.pt")
else:
    print("No data.yaml — skip YOLO; 05_parse_floorplan uses OpenCV fallback")

## Download

Kaggle **Output** tab → download `floorplan_unet.pt` (and `floorplan_yolo.pt` if trained).

Copy to repo `models/` and run:

```powershell
python src/05_parse_floorplan.py data/raw_floorplans/sample.png
```